In [9]:
import os
import json
import glob
import re
import numpy as np
import xarray as xr
from typing import Dict
from joblib import Parallel, delayed
from cftime import num2date, DatetimeNoLeap
from datetime import timedelta
import matplotlib.pyplot as plt

try:
    import xskillscore as xs
except Exception:
    xs = None
try:
    from scipy.stats import t as t_dist
except Exception:
    t_dist = None
    

In [10]:
class ZonalMeanCalculator:
    def __init__(self, regnam, tstart, tend, frequency,
                 model_list, ref_dict, exp_dict, path_in, out_path,
                 var_list=None, force=False,
                 ref_template="monthly/ERA5_analysis_monthly_{year}.nc",
                 mod_template="{year}_*.nc"):
        """
        regnam      : region key (see define_region()).
        tstart,tend : 'YYYY-MM-DD' strings (year/month parsed).
        frequency   : {'monthly','3hourly','6hourly'} (monthly typical).
        model_list  : list of experiment keys.
        ref_dict    : {'ERA5': {'run': '/path/to/ref'}}
        exp_dict    : {'EXP1': {'run': 'CASE_A', 'ref': 'ERA5'}, ...}
        path_in     : model path template with '%(CASENAME)', e.g. '/data/%(CASENAME)/monthly'
        out_path    : root directory for outputs.
        var_list    : e.g. ['U200','T','Z']; None -> defaults from extract_var_list().
        force       : overwrite outputs if True.
        ref_template: file pattern; accepts {year} and/or {var}.
        mod_template: file pattern; accepts {year} and/or {var}.
        """
        self.regnam = regnam
        self.tstart = tstart
        self.tend = tend
        self.frequency = frequency
        self.model_list = model_list
        self.ref_dict = ref_dict
        self.exp_dict = exp_dict
        self.path_in = path_in
        self.out_path = os.path.join(out_path, frequency)
        self.force = force
        self.ref_template = ref_template
        self.mod_template = mod_template

        self.var_dict = self.extract_var_list()
        self.var_list = var_list if var_list is not None else list(self.var_dict.keys())

        self.years = list(range(int(self.tstart[:4]), int(self.tend[:4]) + 1))
        self.seasons = {
            "DJF": ([12, 1, 2], "01-03"),
            "MAM": ([3, 4, 5], "04-06"),
            "JJA": ([6, 7, 8], "07-09"),
            "SON": ([9, 10, 11], "10-12"),
            "ANN": (list(range(1, 13)), "01-12"),
        }
        os.makedirs(self.out_path, exist_ok=True)

    # --------------------------- public API ---------------------------
    def compute(self, pool_across_years=True):
        if pool_across_years:
            for var in self.var_list:
                vinfo = self._require_varinfo(var)
                self._compute_zonal_mean_period(var, vinfo)
        else:
            for year in self.years:
                for var in self.var_list:
                    vinfo = self._require_varinfo(var)
                    self._compute_zonal_mean_year(var, vinfo, year, skip_first_year_djf=True)

    # -------------------------- period-pooled --------------------------
    def _compute_zonal_mean_period(self, var, vinfo):
        varin, vfac = vinfo['alias'], vinfo['fscl']
        season_list = ["DJF", "MAM", "JJA", "SON", "ANN"]
        period_lbl = self._period_label()

        for exp in self.model_list:
            ref = self.exp_dict[exp]['ref']
            out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_zonalmean_{exp}_{period_lbl}.nc")
            if os.path.exists(out_file) and not self.force:
                print(f"Skipping zonal-mean for {var} in {exp} ({period_lbl}) — file exists.")
                continue
            if os.path.exists(out_file) and self.force:
                os.remove(out_file)

            zm_obs_list, zm_fcst_list, nmonths_list = [], [], []

            for season in season_list:
                time_sub = self._get_time_range_for_season_period(season)

                try:
                    obs = self._read_reference_data(
                        ref=ref, period=period_lbl, time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.ref_dict[ref]['run'], template=self.ref_template
                    )[varin].astype("float64")

                    ds = self._read_model_data(
                        exp=exp, period=period_lbl, time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.path_in.replace("%(CASENAME)", self.exp_dict[exp]['run']),
                        template=self.mod_template
                    )
                    obs = self._align_to_times(obs.to_dataset(name=varin), time_sub)[varin]
                    ds  = self._align_to_times(ds, time_sub)
                    fcst = ds[varin].astype("float64")
                except FileNotFoundError:
                    print(f"Data missing for {var}, {exp}, {period_lbl}, {season} — skipping.")
                    # make minimally valid placeholders
                    lat = xr.DataArray(np.full((1,), np.nan), dims=('lat',), coords={'lat':[np.nan]})
                    zm_obs_list.append(xr.full_like(lat, np.nan).rename(f"{var}_zonal_obs").expand_dims(season=[season]))
                    zm_fcst_list.append(xr.full_like(lat, np.nan).rename(f"{var}_zonal_fcst").expand_dims(season=[season]))
                    nmonths_list.append(0)
                    continue

                lat_name = next((d for d in ['lat','latitude','y'] if d in obs.dims), None)
                lon_name = next((d for d in ['lon','longitude','x'] if d in obs.dims), None)
                if lat_name is None or lon_name is None:
                    raise ValueError("Expected lat/lon dims not found in data.")

                # vertical harmonization (supports level-specific names like 'U200')
                obs, fcst, _lev_dim = self._harmonize_vertical(obs, fcst, var, prefer="fcst")

                # chunk fully (safe for dask or numpy)
                obs  = obs.chunk({d: -1 for d in obs.dims})
                fcst = fcst.chunk({d: -1 for d in fcst.dims})

                # time-by-lat zonal means
                obs_zm_t  = obs.mean(lon_name)
                fcst_zm_t = fcst.mean(lon_name)

                # seasonal (time) means → lat (and possibly level) dims remain
                obs_zm  = obs_zm_t.mean('time').rename(f"{var}_zonal_obs")
                fcst_zm = fcst_zm_t.mean('time').rename(f"{var}_zonal_fcst")

                N = int(obs_zm_t.sizes.get('time', 1))
                nmonths_list.append(N)

                zm_obs_list.append(obs_zm.expand_dims(season=[season]))
                zm_fcst_list.append(fcst_zm.expand_dims(season=[season]))

            ds_out = xr.Dataset(
                data_vars=dict(
                    **{f"{var}_zonal_obs":  xr.concat(zm_obs_list,  dim='season')},
                    **{f"{var}_zonal_fcst": xr.concat(zm_fcst_list, dim='season')},
                    n_months = xr.DataArray(np.asarray(nmonths_list), dims="season", coords={"season": season_list}),
                )
            )
            self._annotate_metadata_zm(ds_out, var)
            ds_out.to_netcdf(out_file)
            print(f"Saved: {out_file}")

    # ---------------------------- per-year -----------------------------
    def _compute_zonal_mean_year(self, var, vinfo, year, skip_first_year_djf=True):
        varin, vfac = vinfo['alias'], vinfo['fscl']
        season_full = ["DJF", "MAM", "JJA", "SON", "ANN"]
        season_list = (["MAM", "JJA", "SON", "ANN"]
                       if (skip_first_year_djf and year == self.years[0]) else season_full)

        for exp in self.model_list:
            ref = self.exp_dict[exp]['ref']
            out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_zonalmean_{exp}_{year}.nc")
            if os.path.exists(out_file) and not self.force:
                print(f"Skipping zonal-mean for {var} in {exp} ({year}) — file exists.")
                continue
            if os.path.exists(out_file) and self.force:
                os.remove(out_file)

            zm_obs_list, zm_fcst_list, nmonths_list = [], [], []

            for season in season_list:
                time_sub = (self._get_time_range_for_year(year)
                            if season == "ANN" else self._get_time_range_for_season(year, season))
                try:
                    obs = self._read_reference_data(
                        ref=ref, period=str(year), time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.ref_dict[ref]['run'], template=self.ref_template
                    )[varin].astype("float64")

                    ds = self._read_model_data(
                        exp=exp, period=str(year), time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.path_in.replace("%(CASENAME)", self.exp_dict[exp]['run']),
                        template=self.mod_template
                    )
                    obs = self._align_to_times(obs.to_dataset(name=varin), time_sub)[varin]
                    ds  = self._align_to_times(ds, time_sub)
                    fcst = ds[varin].astype("float64")
                except FileNotFoundError:
                    print(f"Data missing for {var}, {exp}, {year}, {season} — skipping.")
                    lat = xr.DataArray(np.full((1,), np.nan), dims=('lat',), coords={'lat':[np.nan]})
                    zm_obs_list.append(xr.full_like(lat, np.nan).rename(f"{var}_zonal_obs").expand_dims(season=[season]))
                    zm_fcst_list.append(xr.full_like(lat, np.nan).rename(f"{var}_zonal_fcst").expand_dims(season=[season]))
                    nmonths_list.append(0)
                    continue

                lat_name = next((d for d in ['lat','latitude','y'] if d in obs.dims), None)
                lon_name = next((d for d in ['lon','longitude','x'] if d in obs.dims), None)
                if lat_name is None or lon_name is None:
                    raise ValueError("Expected lat/lon dims not found in data.")

                obs  = obs.chunk({lat_name: -1, lon_name: -1, 'time': -1})
                fcst = fcst.chunk({lat_name: -1, lon_name: -1, 'time': -1})

                # If var has pressure levels (or a target level like U200 was *not* requested),
                # harmonize vertical coordinates so obs and model share the same level grid.
                obs, fcst, _lev_dim = self._harmonize_vertical(obs, fcst, var, prefer="fcst")

                obs_zm_t  = obs.mean(lon_name)
                fcst_zm_t = fcst.mean(lon_name)
                obs_zm  = obs_zm_t.mean('time').rename(f"{var}_zonal_obs")
                fcst_zm = fcst_zm_t.mean('time').rename(f"{var}_zonal_fcst")
                N = int(obs_zm_t.sizes.get('time', 1))

                zm_obs_list.append(obs_zm.expand_dims(season=[season]))
                zm_fcst_list.append(fcst_zm.expand_dims(season=[season]))
                nmonths_list.append(N)

            ds_out = xr.Dataset(
                data_vars=dict(
                    **{f"{var}_zonal_obs":  xr.concat(zm_obs_list,  dim='season')},
                    **{f"{var}_zonal_fcst": xr.concat(zm_fcst_list, dim='season')},
                    n_months = xr.DataArray(np.asarray(nmonths_list), dims="season", coords={"season": season_list}),
                )
            )
            self._annotate_metadata_zm(ds_out, var)
            ds_out.to_netcdf(out_file)
            print(f"Saved: {out_file}")

    # --------------------------- time helpers ---------------------------
    def _period_label(self):
        return f"{self.tstart[:7].replace('-','')}_{self.tend[:7].replace('-','')}"

    def _all_month_starts(self):
        y0, m0 = int(self.tstart[:4]), int(self.tstart[5:7])
        y1, m1 = int(self.tend[:4]),   int(self.tend[5:7])
        dates = []
        y, m = y0, m0
        while (y < y1) or (y == y1 and m <= m1):
            dates.append(DatetimeNoLeap(y, m, 1))
            m += 1
            if m == 13: m = 1; y += 1
        return xr.CFTimeIndex(dates)

    def _get_time_range_for_season_period(self, season):
        months, _ = self.seasons[season]
        all_months = self._all_month_starts()
        if season == "ANN":
            return all_months
        return xr.CFTimeIndex([d for d in all_months if d.month in months])

    def _get_time_range_for_year(self, year):
        freq_map = {"3hourly": "3h", "6hourly": "6h", "monthly": "1MS"}
        return xr.cftime_range(f"{year}-01-01", f"{year}-12-31",
                               freq=freq_map.get(self.frequency, "1MS"),
                               calendar="noleap")

    def _get_time_range_for_season(self, year, season):
        months, _ = self.seasons[season]
        dates = []
        for m in months:
            y = year if not (season == "DJF" and m == 12) else year - 1
            dates.append(DatetimeNoLeap(y, m, 1))
        return xr.CFTimeIndex(sorted(dates))

    def _years_from_time_sub(self, time_sub, fallback_year=None):
        years = set()
        if isinstance(time_sub, slice):
            for endpoint in (time_sub.start, time_sub.stop):
                if hasattr(endpoint, "year"): years.add(int(endpoint.year))
        else:
            try:
                for t in time_sub:
                    if hasattr(t, "year"): years.add(int(t.year))
            except TypeError:
                if hasattr(time_sub, "year"): years.add(int(time_sub.year))
        if not years and fallback_year is not None:
            years = {int(fallback_year)}
        return sorted(years)

    # ----------------------- IO + harmonization -----------------------
    def _read_model_data(self, exp, period, time_sub, regnam, var, vfac, data_dir, template, diag_print=False):
        years_needed = self._years_from_time_sub(time_sub, fallback_year=period)
        base, target_plev = self._parse_varname(var)
        if base == "Z": base = 'Z3'

        paths = []
        for y in years_needed:
            patt = self._render_pattern(template, y, var=base)
            pattern = os.path.join(data_dir, patt)
            matches = sorted(glob.glob(pattern))
            if not matches: print(f"[WARN] Model no matches: {pattern}")
            paths.extend(matches)
        if not paths:
            raise FileNotFoundError(f"No model files found for years={years_needed}")

        dm = xr.open_mfdataset(paths, combine="by_coords")

        (lat_bnds, lon_bnds) = self.define_region(regnam)
        lat_name = next((d for d in ['lat','latitude','y'] if d in dm.coords), None)
        lon_name = next((d for d in ['lon','longitude','x'] if d in dm.coords), None)
        if lat_name is None or lon_name is None:
            raise ValueError("Model data missing lat/lon coordinates.")

        dm = self._select_level(dm, target_plev)
        dm = self._maybe_wrap_lon(dm, lon_name)

        # region bounds
        global_like = (lon_bnds[0] <= -179.999) and (lon_bnds[1] >= 179.999)

        dm = self._set_time_to_midpoint(dm)
        dm = self._normalize_time_to_month_start(dm)
        dm = dm.convert_calendar("noleap", use_cftime=True)
        dm = self._align_to_times(dm, time_sub)

        dm = self._smart_lat_slice(dm, lat_bnds, lat_name)
        if not global_like:
            dm = dm.sel({lon_name: slice(*lon_bnds)})

        if base in dm:
            dm[base] = self.apply_unit_scaling_mod(base, dm[base], vfac)
        if diag_print and "time" in dm.dims:
            print(f"[DIAG] Model Time: {str(dm.time.min().values)} → {str(dm.time.max().values)} (n={dm.sizes['time']})")
        return dm

    def _read_reference_data(self, ref, period, time_sub, regnam, var, vfac, data_dir, template, diag_print=False):
        years_needed = self._years_from_time_sub(time_sub, fallback_year=period)
        base, target_plev = self._parse_varname(var)
        if base == "Z": base = 'Z3'

        paths = []
        for y in years_needed:
            patt = self._render_pattern(template, y, var=base)
            p = os.path.join(data_dir, patt)
            matches = sorted(glob.glob(p))
            if not matches: print(f"[WARN] Reference no matches: {p}")
            paths.extend(matches)
        if not paths:
            raise FileNotFoundError(f"No reference files found for years={years_needed}")

        dr = xr.open_mfdataset(paths, combine="by_coords")

        # normalize coord names
        rename_dict = {}
        if "longitude" in dr.dims and "lon" not in dr.dims: rename_dict["longitude"] = "lon"
        if "latitude"  in dr.dims and "lat" not in dr.dims: rename_dict["latitude"]  = "lat"
        if rename_dict: dr = dr.rename(rename_dict)

        lat_name = next((d for d in ['lat','latitude','y'] if d in dr.coords), None)
        lon_name = next((d for d in ['lon','longitude','x'] if d in dr.coords), None)
        if lat_name is None or lon_name is None:
            raise ValueError("Reference data missing lat/lon coordinates.")

        dr = self._select_level(dr, target_plev)
        dr = self._maybe_wrap_lon(dr, lon_name)

        (lat_bnds, lon_bnds) = self.define_region(regnam)
        global_like = (lon_bnds[0] <= -179.999) and (lon_bnds[1] >= 179.999)

        dr = dr.convert_calendar("noleap", use_cftime=True)
        dr = self._normalize_time_to_month_start(dr)

        # apply time selection
        if isinstance(time_sub, slice):
            dr = dr.sel(time=time_sub)
        else:
            try:
                dr = dr.sel(time=slice(*time_sub))
            except Exception:
                dr = dr.sel(time=time_sub)

        dr = self._smart_lat_slice(dr, lat_bnds, lat_name)
        if not global_like:
            dr = dr.sel({lon_name: slice(*lon_bnds)})

        if base in dr:
            dr[base] = self.apply_unit_scaling_obs(base, dr[base], vfac)
        if diag_print and "time" in dr.dims:
            print(f"[DIAG] OBS Time: {str(dr.time.min().values)} → {str(dr.time.max().values)} (n={dr.sizes['time']})")
        return dr

    # -------------------------- helpers & math --------------------------
    def _require_varinfo(self, var):
        if var not in self.var_dict:
            raise ValueError(f"Variable '{var}' is not defined in var_dict.")
        return self.var_dict[var]

    @staticmethod
    def _parse_varname(varname):
        m = re.match(r"([A-Za-z]+)(\d+)$", varname)
        if m:
            return m.group(1), int(m.group(2)) * 100  # hPa → Pa
        return varname, None

    @staticmethod
    def _lev_name(da_or_ds):
        for k in ("plev", "lev", "level"):
            if k in getattr(da_or_ds, "dims", ()) or k in getattr(da_or_ds, "coords", {}):
                return k
        return None

    @staticmethod
    def _levels_to_hpa(coord):
        vals = np.array(coord.values, dtype=float)
        units = (getattr(coord, "attrs", {}) or {}).get("units", "") or str(getattr(coord, "units", ""))
        u = units.lower()
        if "pa" in u and "hpa" not in u:
            return vals / 100.0, "hPa"
        if np.nanmax(np.abs(vals)) > 2000:
            return vals / 100.0, "hPa"
        return vals, "hPa"

    def _harmonize_vertical(self, obs, fcst, var, *, prefer="obs", tol_hpa=0.05):
        base, target_plev = self._parse_varname(var)
        if target_plev is not None:
            return obs, fcst, None

        lo = self._lev_name(obs)
        lf = self._lev_name(fcst)
        if lo is None and lf is None:
            return obs, fcst, None
        if (lo is None) ^ (lf is None):
            side = "obs" if lo is not None else "fcst"
            raise ValueError(
                f"Vertical mismatch: {side} has a level dimension but the other does not. "
                f"Use a level-specific variable name (e.g., '{base}200') or ensure both inputs carry comparable levels."
            )
        if lo != lf:
            fcst = fcst.rename({lf: lo})
            lf = lo

        obs_hpa, _ = self._levels_to_hpa(obs[lo])
        fcst_hpa, _ = self._levels_to_hpa(fcst[lo])
        obs = obs.assign_coords({lo: obs_hpa})
        fcst = fcst.assign_coords({lo: fcst_hpa})

        lo_min = max(np.nanmin(obs_hpa),  np.nanmin(fcst_hpa))
        lo_max = min(np.nanmax(obs_hpa),  np.nanmax(fcst_hpa))
        target = (obs_hpa if prefer == "obs" else fcst_hpa)
        target = target[(target >= lo_min - tol_hpa) & (target <= lo_max + tol_hpa)]
        if target.size == 0:
            raise ValueError("No overlapping pressure levels between obs and fcst.")
        if prefer == "obs":
            fcst = fcst.interp({lo: target})
            obs  = obs.sel({lo: target})
        else:
            obs  = obs.interp({lo: target})
            fcst = fcst.sel({lo: target})

        obs[lo].attrs["units"]  = "hPa"
        fcst[lo].attrs["units"] = "hPa"
        return obs, fcst, lo

    @staticmethod
    def _set_time_to_midpoint(ds):
        if 'time_bnds' in ds.variables and 'time' in ds.coords:
            mid = ds['time_bnds'].mean(dim=ds['time_bnds'].dims[-1])
            ds = ds.assign_coords(time=mid)
        return ds

    @staticmethod
    def _normalize_time_to_month_start(ds):
        if "time" not in ds.coords: return ds
        new_times = []
        for t in ds["time"].values:
            t_str = str(type(t))
            if "datetime64" in t_str:
                import pandas as pd
                ts = pd.to_datetime(t)
                new_times.append(ts.to_period('M').to_timestamp('MS').to_pydatetime())
            elif "Timestamp" in t_str:
                new_times.append(t.to_period('M').to_timestamp('MS').to_pydatetime())
            else:
                cls = t.__class__
                new_times.append(cls(t.year, t.month, 1))
        return ds.assign_coords(time=("time", new_times))

    def _align_to_times(self, ds, target_index, *, tolerance_days=3):
        if "time" not in ds.coords:
            return ds
        ds2 = ds.reindex(
            time=target_index,
            method="nearest",
            tolerance=np.timedelta64(tolerance_days, "D")
        )
        if np.isnan(ds2["time"].astype("datetime64[ns]").values).any():
            raise ValueError("[ERROR] Failed strict month-start alignment (missing after reindex).")
        return ds2

    @staticmethod
    def _to_m180_180(lon):
        return ((lon + 180) % 360) - 180

    def _maybe_wrap_lon(self, ds, lon_name):
        lon = ds[lon_name]
        if lon.min() >= 0 and lon.max() <= 360:
            ds = ds.assign_coords({lon_name: self._to_m180_180(lon)}).sortby(lon_name)
        return ds

    def _smart_lat_slice(self, arr, lat_bnds, lat_name):
        lo, hi = lat_bnds
        lat = arr[lat_name]
        asc = bool(lat.values[0] < lat.values[-1])
        return arr.sel({lat_name: slice(lo, hi) if asc else slice(hi, lo)})
        
    @staticmethod
    def _select_level(dr, target_plev):
        """
        If target_plev is provided (in Pa), reduce/interp the dataset to that pressure level.
        Handles level coords named 'plev'/'lev'/'level', with units in Pa or hPa.
        Falls back to nearest index if interpolation is not possible.
        """
        if target_plev is None:
            return dr
        for lev_dim in ["plev", "lev", "level"]:
            if lev_dim in dr.dims:
                lev = dr[lev_dim]
                vals = lev.values
                units = (getattr(lev, "attrs", {}) or {}).get("units", "") or str(getattr(lev, "units", ""))
                units = units.lower()

                # normalize to hPa values
                if "pa" in units and "hpa" not in units:
                    vals = vals / 100.0
                elif np.nanmax(vals) > 2000:
                    vals = vals / 100.0

                target_hpa = target_plev / 100.0

                # try linear interpolation in pressure space
                if np.size(vals) >= 2 and np.isfinite(vals).all():
                    dr = dr.assign_coords({lev_dim: vals})
                    try:
                        return dr.interp({lev_dim: target_hpa})
                    except Exception:
                        pass  # fall through to nearest

                # nearest level fallback
                idx = int(np.nanargmin(np.abs(vals - target_hpa)))
                return dr.isel({lev_dim: idx}, drop=True)
        return dr

    @staticmethod
    def apply_unit_scaling_obs(var, da, vfac):
        if var in ['Z', 'Z3']:
            return da * vfac / 9.80616
        elif var in ['SHFLX', 'TAUX', 'TAUY']:
            return da * vfac * -1.0
        elif var == 'LHFLX':
            return da * vfac * -2.501e6
        elif var in ['T', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        elif var == 'PRECT':
            return da * vfac / 3600.0 * 1000.0 * 86400.0
        else:
            return da * vfac

    @staticmethod
    def apply_unit_scaling_mod(var, da, vfac):
        if var in ['Z', 'Z3']:
            return da * vfac / 9.80616
        elif var == 'PRECT':
            return da * vfac * 1000.0 * 86400.0
        elif var == 'LHFLX':
            return da * vfac * -2.501e6
        elif var in ['T', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        else:
            return da * vfac

    @staticmethod
    def define_region(regnam='global'):
        reg_dict = {
            'global':   [(-90, 90),  (-180, 180)],
            'Atlantic': [(5, 55),    (-95, -40)],
            'CONUS':    [(25, 50),   (-125, -95)],
            'Antarctic':[(-90, -50), (-180, 180)],
            'PolarN':   [(50, 90),   (-180, 180)],
            'Greenland':[(60, 85),   (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Unknown region '{regnam}'. Valid: {list(reg_dict)}")
        return reg_dict[regnam]

    def extract_var_list(self):
        return {
            'U': {'alias': 'U',  'unit': 'm s$^{-1}$', 'fscl': 1.0, 'min': -0.2,  'max': 0.2,   'nlev': 11},
            'V': {'alias': 'V',  'unit': 'm s$^{-1}$', 'fscl': 1.0, 'min': -0.2,  'max': 0.2,   'nlev': 11},
            'T': {'alias': 'T',  'unit': '°C',         'fscl': 1.0, 'min': -10,   'max': 10,    'nlev': 11},
            'Q': {'alias': 'Q',  'unit': 'kg kg$^{-1}$','fscl':1.0, 'min': -2e-3, 'max': 2e-3,  'nlev': 11},
            'Z': {'alias': 'Z3', 'unit': 'm',          'fscl': 1.0, 'min': -200,  'max': 200,   'nlev': 11},
        }

    # --- rendering helper for filename patterns ---
    @staticmethod
    def _render_pattern(template, year, var=None):
        try:
            return template.format(year=year, var=var)
        except Exception:
            try:
                return template.format(year)
            except Exception:
                patt = template.replace("%Y", str(year))
                if var is not None:
                    patt = patt.replace("%VAR%", str(var))
                return patt

    @staticmethod
    def _annotate_metadata_zm(ds_out, var):
        ds_out[f'{var}_zonal_obs'].attrs['long_name']  = f"Seasonal zonal-mean {var} (obs)"
        ds_out[f'{var}_zonal_fcst'].attrs['long_name'] = f"Seasonal zonal-mean {var} (model)"
        ds_out['n_months'].attrs['description']        = "Number of monthly samples used in the season"

In [11]:
if __name__ == "__main__":
    # --- paths ---
    top_path  = "/pscratch/sd/z/zhan391/seacrogs_scratch"
    data_path = f"{top_path}/post_data"
    out_path  = "/pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean"
    os.makedirs(out_path, exist_ok=True)

    # --- experiments metadata ---
    exp_json = f"{data_path}/scripts/ml_exp_info.json"
    with open(exp_json, "r") as f:
        exp_dict = json.load(f)

    # ensure each experiment has a 'ref' key (default to ERA5 if missing)
    for k, v in exp_dict.items():
        v.setdefault("ref", "ERA5")

    # --- time / freq / region ---
    tstart = "2012-01-01"
    tend   = "2016-12-31"
    freq   = "monthly"
    regnam = "global"

    # --- reference dataset info ---
    ref_dict = {
        "ERA5": {
            "run": f"{data_path}/ERA5",      # directory containing ERA5 files
            "period": "200801_201712",       # (not required by the class, kept for provenance)
        }
    }

    # --- model file root ---
    # The class will look under: path_in.replace("%(CASENAME)", exp_dict[exp]["run"])
    path_template = f"{data_path}/%(CASENAME)/{freq}"

    # Optionally customize filename patterns (defaults shown here):
    ref_template = "monthly/ERA5_analysis_monthly_{year}.nc"  # accepts {year} and/or {var}
    mod_template = "{year}_*.nc"                              # accepts {year} and/or {var}

    # --- variables ---
    # None → use defaults from extract_var_list(); or set e.g. ['U200','T','Z']
    variables = None

    # --- init calculator ---
    calculator = ZonalMeanCalculator(
        regnam=regnam,
        tstart=tstart,
        tend=tend,
        frequency=freq,
        model_list=list(exp_dict.keys()),
        ref_dict=ref_dict,
        exp_dict=exp_dict,
        path_in=path_template,
        out_path=out_path,
        var_list=variables,
        force=True,
        ref_template=ref_template,
        mod_template=mod_template,
    )

    # --- run ---
    # pool_across_years=True → single file per exp with seasonal means over 2012–2014
    calculator.compute(pool_across_years=True)
    

Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean/monthly/U_global_zonalmean_CLIM_201201_201612.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean/monthly/U_global_zonalmean_UNet_201201_201612.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean/monthly/U_global_zonalmean_UNetMP_201201_201612.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean/monthly/U_global_zonalmean_IUNet_201201_201612.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean/monthly/U_global_zonalmean_MnM_201201_201612.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean/monthly/V_global_zonalmean_CLIM_201201_201612.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_mean/monthly/V_global_zonalmean_UNet_201201_